In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develop a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import numpy as np
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"  # "20251028_140930" # "20251027_152036"

In [ ]:
"""--------------------------------------------"""
# add in model parameters to encoder, look at cvr2
# cvr2 with strategy model params
# reward prediction error (MF), q-learner, need to get q-value
"""---------------------------------------------"""
# build in interaction terms and see what pops out in the cvr2/dr2 plots
"""---------------------------------------------"""
# add movement over time
"""---------------------------------------------"""
# cvr2 across time
# --> when does encoding emerge across time?
"""---------------------------------------------"""
# aggregate across sessions
"""---------------------------------------------"""
# check for temporal autocorrelations by ensuring no encoding after shuffling trial data.
# incorporate as a sanity check to pass for all sessions

# > check the fits for different regularization constants
# define responsive
"""--------------------------------------------"""

In [ ]:
# TODO
# cvr2/dr2 with bswitch and interaction terms
# plot bswitch bweight over time

## init

In [ ]:
from sg.models import Encoder, StrategyEncoder

encoder = Encoder(
    subj_id,
    sess_id,
    tv_keys=[
        "response",
        "rewarded",
        "response_prev",
        "rewarded_prev",
    ],
    binwidth_ms=100,
    add_interaction=True,
    norm=True,
)
encoder.verify()

encoder_mb = StrategyEncoder(
    subj_id,
    sess_id,
    tv_keys=["response", "rewarded", "response_prev", "rewarded_prev"],
    binwidth_ms=100,
    norm=True,
    strategy_filter="mb",
)
encoder_mf = StrategyEncoder(
    subj_id,
    sess_id,
    tv_keys=["response", "rewarded", "response_prev", "rewarded_prev"],
    binwidth_ms=100,
    norm=True,
    strategy_filter="mf",
)

encoder_mb.verify()
encoder_mf.verify()

In [ ]:
encoder.dm_idxs

In [ ]:
from sg.models import ShuffledEncoder

se = ShuffledEncoder(
    subj_id,
    sess_id,
    tv_keys=[
        "response",
        "rewarded",
        "response_prev",
        "rewarded_prev",
    ],
    add_interaction=True,
)
se.plot_cvr2()
se.plot_dr2()
se.plot_bound_r2()

## akam 2021 fig 4

In [ ]:
reg = "DLS"

trial_type = {
    "lc": (encoder.trial_data.response == 1) & (encoder.trial_data.rewarded == 1),
    "rc": (encoder.trial_data.response == -1) & (encoder.trial_data.rewarded == 1),
    "li": (encoder.trial_data.response == 1) & (encoder.trial_data.rewarded == 0),
    "ri": (encoder.trial_data.response == -1) & (encoder.trial_data.rewarded == 0),
}

psths_tt = {
    k: encoder.psths[reg][:, mask, :].mean(axis=1) for k, mask in trial_type.items()
}

In [ ]:
tt_mean = np.mean([psth for psth in psths_tt.values()], axis=0)

psths_tt = {k: psth - tt_mean for k, psth in psths_tt.items()}

In [ ]:
# [n_neurons, n_tt*n_tpoints]

psths_tt_mat = np.concatenate([psth.T for psth in psths_tt.values()])
print(psths_tt_mat.shape)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA().fit(psths_tt_mat)

plt.figure(tight_layout=True)
plt.plot(
    [np.sum(pca.explained_variance_ratio_[:i]) for i in range(psths_tt_mat.shape[0])]
)
plt.axhline(y=0.9, linestyle="--", color="#666666")
plt.xlabel("n. components")
plt.ylabel("explained variance ratio")
plt.show()

In [ ]:
psths_proj = pca.transform(psths_tt_mat)[:, :3]
psths_proj.shape

In [ ]:
n_tpoints = encoder.psths[reg].shape[-1]
psths_tt_proj = {
    k: psths_proj[i * n_tpoints : (i + 1) * n_tpoints, :]
    for i, k in enumerate(trial_type.keys())
}

In [ ]:
encoder.tbin_centers

In [ ]:
from core.viz import plot_trajectory
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors


def truncate_colormap(cmap, minval=0.35, maxval=1.0, n=256):
    if isinstance(cmap, str):
        cmap = plt.get_cmap(cmap)
    return mcolors.LinearSegmentedColormap.from_list(
        f"trunc({cmap.name},{minval:.2f},{maxval:.2f})",
        cmap(np.linspace(minval, maxval, n)),
    )


def plot_trajectories(
    psths, pc_a=0, pc_b=1, cmaps=None, cmap_min=0.35, ax=None, pad=0.05, **traj_kwargs
):
    if ax is None:
        _, ax = plt.subplots(figsize=(3, 3), tight_layout=True)

    all_xy = np.concatenate(list(psths.values()), axis=0)
    lim = np.abs(all_xy).max() * (1 + pad)

    default_cmaps = ["YlGn", "Blues", "OrRd", "RdPu"]
    if cmaps is None:
        cmaps = {k: default_cmaps[i] for i, k in enumerate(psths)}
    cmaps = {k: truncate_colormap(c, minval=cmap_min) for k, c in cmaps.items()}

    handles = []
    for k, psth in psths.items():
        plot_trajectory(
            psth[:, pc_a],
            psth[:, pc_b],
            xlabel=f"pc {pc_a + 1}",
            ylabel=f"pc {pc_b + 1}",
            cmap=cmaps[k],
            mn=-lim,
            mx=lim,
            ax=ax,
            **traj_kwargs,
        )
        handles.append(Line2D([0], [0], color=cmaps[k](0.6), lw=2, label=k))

    ax.legend(handles=handles)

    return ax


ax = plot_trajectories(psths_tt_proj, pc_a=1, pc_b=2)

In [ ]:
fig, ax = plt.subplots()

for k, psth in psths_tt_proj.items():
    ax.plot(psth[:, 0], psth[:, 1], label=k)
fig.legend()

## psths

In [ ]:
from scipy.stats import zscore

reg = "DLS"


def plot_peths(reg="DLS", sort_by="peak_fr_time"):
    if sort_by == "peak_fr_time":
        psths_reg = encoder.psths[reg].mean(axis=1)
        psths_reg = zscore(psths_reg, axis=1)
        peak_fr_sort_idxs = np.argsort(
            [np.argsort(psths_reg[i])[-1] for i in range(len(psths_reg))]
        )

        plt.figure(figsize=(3, 4))
        plt.imshow(psths_reg[peak_fr_sort_idxs], cmap="magma")
        plt.xticks(
            np.arange(psths_reg.shape[1], step=10),
            np.round(encoder.tbin_edges[:-1:10], 2),
        )
        plt.colorbar(shrink=0.5)
        plt.title(f"{reg} PETHs (s.b. peak fr time)", fontsize=7)

        plt.show()


plot_peths(reg="DLS")
plot_peths(reg="DMS")

In [ ]:
from scipy.stats import pearsonr as r

bweight_diff = encoder_mb.encoder_weights - encoder_mf.encoder_weights
bweight_diff = bweight_diff[:, encoder.num_tents :]

diff_corr = np.array(
    [
        [
            r(bweight_diff[:, i], bweight_diff[:, j]).statistic
            for i in range(bweight_diff.shape[1])
        ]
        for j in range(bweight_diff.shape[1])
    ]
)
diff_corr_reg = {
    reg: np.array(
        [
            [
                r(
                    bweight_diff[encoder.reg_idxs[reg], i],
                    bweight_diff[encoder.reg_idxs[reg], j],
                ).statistic
                for i in range(bweight_diff.shape[1])
            ]
            for j in range(bweight_diff.shape[1])
        ]
    )
    for reg in encoder.regions
}

In [ ]:
def plot_corr(diff_corr, reg=None):
    if reg is not None:
        diff_corr = diff_corr[reg]
    fig, ax = plt.subplots(figsize=(5, 5))

    ax.imshow(diff_corr, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(len(diff_corr)), encoder.tv_keys, rotation=90)
    ax.set_yticks(range(len(diff_corr)), encoder.tv_keys)

    title = (
        rf"$r$ of bweight difference between strategies, {reg}"
        if reg is not None
        else r"$r$ of bweight difference between strategies"
    )
    fig.suptitle(title)


plot_corr(diff_corr)
plot_corr(diff_corr_reg, reg="DLS")
plot_corr(diff_corr_reg, reg="DMS")

In [ ]:
from core.data import get_strategy_filter_idxs


def build_encoder_strategy_ctrl(strategy="mb"):
    encoder = Encoder(
        subj_id,
        sess_id,
    )
    encoder.get_data()

    # get an even percentage of model-based and model-free
    idxs = get_strategy_filter_idxs(
        encoder.trial_data, balance_strategy=True, cond_balance=False
    )
    idxs_mb = np.random.choice(idxs["mb"], len(idxs["mb"]) // 2)
    idxs_mf = np.random.choice(idxs["mf"], len(idxs["mf"]) // 2)
    idxs = np.sort(np.concatenate((idxs_mb, idxs_mf)))

    encoder_strategy = StrategyEncoder(
        subj_id,
        sess_id,
        idxs=idxs,
        tv_keys=["response", "rewarded", "response_prev", "rewarded_prev"],
        strategy_filter=strategy,
    )
    encoder_strategy.fit_encoder()

    assert len(np.unique(encoder_strategy.trial_data["strategy"])) == 2
    assert np.isclose(
        (encoder_strategy.trial_data["strategy"] == 1).mean(), 0.5, atol=0.15
    ), (encoder_strategy.trial_data["strategy"] == 1).mean()

    return encoder_strategy


encoder_mb_ctrl = build_encoder_strategy_ctrl(strategy="mb")
encoder_mf_ctrl = build_encoder_strategy_ctrl(strategy="mf")

In [ ]:
bweight_diff_ctrl = encoder_mb_ctrl.encoder_weights - encoder_mf_ctrl.encoder_weights
bweight_diff_ctrl = bweight_diff_ctrl[:, encoder.num_tents :]

diff_corr_ctrl = np.array(
    [
        [
            r(bweight_diff_ctrl[:, i], bweight_diff_ctrl[:, j]).statistic
            for i in range(bweight_diff_ctrl.shape[1])
        ]
        for j in range(bweight_diff_ctrl.shape[1])
    ]
)
diff_corr_ctrl_reg = {
    reg: np.array(
        [
            [
                r(
                    bweight_diff_ctrl[encoder.reg_idxs[reg], i],
                    bweight_diff_ctrl[encoder.reg_idxs[reg], j],
                ).statistic
                for i in range(bweight_diff_ctrl.shape[1])
            ]
            for j in range(bweight_diff_ctrl.shape[1])
        ]
    )
    for reg in encoder.regions
}

In [ ]:
plot_corr(diff_corr_ctrl)
plot_corr(diff_corr_ctrl_reg, reg="DLS")
plot_corr(diff_corr_ctrl_reg, reg="DMS")

In [ ]:
diff_corr_emc = diff_corr - diff_corr_ctrl
diff_corr_emc_reg = {
    reg: diff_corr_reg[reg] - diff_corr_ctrl_reg[reg] for reg in encoder.regions
}

plot_corr(diff_corr_emc)
plot_corr(diff_corr_emc_reg, reg="DLS")
plot_corr(diff_corr_emc_reg, reg="DMS")

## weight correlation

In [ ]:
from core.data import tv_vals
from core.viz import plot_kdes

weight_diff = {}
for regr in encoder.tv_keys:
    if regr != "response_prev":
        regr_ = f"{regr}_{tv_vals[regr][0]}"
        weight_diff[regr_] = (
            encoder_mb.encoder_weights[:, encoder.dm_idxs[regr_]]
            - encoder_mf.encoder_weights[:, encoder.dm_idxs[regr_]]
        )

plot_kdes(weight_diff)

## pca on DMS/DLS robs

In [ ]:
import numpy as np

In [ ]:
from sklearn.decomposition import PCA
from utils.colors import colors_region

pca_dls = PCA().fit(encoder.robs[:, encoder.reg_idxs["DLS"]])
pca_dms = PCA().fit(encoder.robs[:, encoder.reg_idxs["DMS"]])
cum_var_dls = np.array([sum(pca_dls.explained_variance_ratio_[:n]) for n in range(100)])
cum_var_dms = np.array([sum(pca_dms.explained_variance_ratio_[:n]) for n in range(100)])

plt.figure(tight_layout=True)
plt.plot(
    cum_var_dls,
    color=colors_region["DLS"],
    label=f"DLS (n={np.where(cum_var_dls > 0.9)[0][0]})",
)
plt.plot(
    cum_var_dms,
    color=colors_region["DMS"],
    label=f"DMS (n={np.where(cum_var_dms > 0.9)[0][0]})",
)
plt.legend()
plt.axhline(y=0.9, color="#666666", linestyle="--")
plt.xlabel("n. components")
plt.ylabel("p(explained variance)")
plt.title("spike counts")
plt.show()

## pca based on weights

In [ ]:
from sklearn.decomposition import PCA

pca_dls = PCA().fit(encoder.encoder_weights[encoder.reg_idxs["DLS"]])
pca_dms = PCA().fit(encoder.encoder_weights[encoder.reg_idxs["DMS"]])

cum_var_dls = np.array(
    [
        sum(pca_dls.explained_variance_ratio_[:n])
        for n in range(encoder.num_tents + encoder.num_tv)
    ]
)
cum_var_dms = np.array(
    [
        sum(pca_dms.explained_variance_ratio_[:n])
        for n in range(encoder.num_tents + encoder.num_tv)
    ]
)

plt.figure(tight_layout=True)
plt.plot(
    cum_var_dls,
    color=colors_region["DLS"],
    label=f"DLS (n={np.where(cum_var_dls > 0.9)[0][0]})",
)
plt.plot(
    cum_var_dms,
    color=colors_region["DMS"],
    label=f"DMS (n={np.where(cum_var_dms > 0.9)[0][0]})",
)
plt.axhline(y=0.9, color="#666666", linestyle="--")
plt.xlabel("n. components")
plt.ylabel("p(explained variance)")
plt.title("encoding beta weights")
plt.legend()
plt.show()

In [ ]:
pca = PCA().fit(encoder.encoder_weights)
cum_var = np.array(
    [
        sum(pca.explained_variance_ratio_[:n])
        for n in range(encoder.num_tents + encoder.num_tv)
    ]
)

plt.figure(tight_layout=True)
plt.plot(cum_var)
plt.axhline(y=0.9, color="#666666", linestyle="--")
plt.xlabel("n. components")
plt.ylabel("p(explained variance)")
plt.show()

In [ ]:
n = np.where(cum_var >= 0.9)[0][0]
pca = PCA(n_components=n).fit(encoder.encoder_weights)
weights_lowd = pca.transform(encoder.encoder_weights)

In [ ]:
weights_lowd[:, :3]

In [ ]:
plt.figure()
plt.scatter(weights_lowd[:, 0], weights_lowd[:, 1], alpha=0.5, s=0.5)
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection="3d")

ax.scatter(xs=weights_lowd[:, 0], ys=weights_lowd[:, 1], zs=weights_lowd[:, 2], s=0.3)
plt.show()